# 03 · Unit, prediction point, predictors and gates G2 and G3 — Neonatal mortality NCHS (M4)

| Field | Content |
|---|---|
| Framework sequence | 01: S1 · S2 · S3 → 02: S4 · G1 → **03: S5 · S6 · S7 · G2 · G3** → 04: S8 → 05: S8 · S9 · G4 → 06: S10 · S11 |
| What it does | Builds the contract table with the 71 parameters known at birth + 5 minutes, checks for leakage and checks that there are enough events with the Riley et al. criteria |
| Stages and gates | E5 · E6 · E7 · P2 · P3 |
| Decisions applied | M4-D1 (48-hour label), M4-D2 (no marital status), M4-D3 (unknown = missing, no indicators) |
| Inputs | `data/interim/cohort_<year>.parquet` |
| Outputs | `data/contract/neonatal_nchs_v1/cohort_<year>.parquet` (not versioned) and `data/outputs_03_predictors_g2_g3.txt` |
| Time and memory | ≈10 minutes the first time (≈6 GB); later the contract table is reused |
| Previous / next | `02_count_audit` / `04_preliminary_learners` |

> **Rebuilding the contract table.** If it already exists in `data/contract/neonatal_nchs_v1/`, it is reused. To build it again (for example, if the predictors change or the cohorts of notebook 01 were rebuilt), set **`REBUILD = True`** in the first code cell.

In [ ]:
import os, sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from libraries.nchs_reader import COHORTS, OUT_DIR

# Language of the outputs: "es" in the Spanish notebook, "en" in the English one (the only line that differs).
LANG = "en"
import importlib
import libraries.i18n_es, libraries.i18n          # reloaded so that a running kernel picks up dictionary changes
importlib.reload(libraries.i18n_es); importlib.reload(libraries.i18n)
from libraries.i18n import translate
pd.set_option("display.max_colwidth", None)      # decision texts are shown in full
import warnings
# scikit-learn >= 1.8 announces that LogisticRegression(penalty=...) will be replaced by l1_ratio. The models keep
# penalty (the hyperparameters chosen in notebook 04) so that results stay exactly as reported; the notice is hidden.
warnings.filterwarnings("ignore", message=".*'penalty' was deprecated.*", category=FutureWarning)
lines_es = []
if LANG == "es":                       # progress messages (print) also in Spanish
    import builtins
    def print(*args, **kwargs):
        builtins.print(*[translate(a, "es") if isinstance(a, str) and not a.startswith("\n== ") else a for a in args], **kwargs)

def write_outputs(path, header):
    # English text output (cited by the documents) and, from the Spanish notebook, a Spanish copy (*_es.txt).
    path.write_text(header + "\n" + "\n".join(lines) + "\n", encoding="utf-8")
    if LANG == "es":
        path.with_name(path.stem + "_es.txt").write_text(translate(header, "es") + "\n" + "\n".join(lines_es) + "\n", encoding="utf-8")


CONTRACT_DIR = Path(os.environ.get("NCHS_CONTRACT_DIR", ROOT / "data" / "contract" / "neonatal_nchs_v1"))
OUT_FILE = ROOT / "data" / "outputs_03_predictors_g2_g3.txt"
DEV = [y for y in COHORTS if y <= 2021]          # temporal test: cohorts 2022-2023 (criterion C4)
SEED = 20261123
lines = []

def show(title, obj):
    txt = obj.to_string() if hasattr(obj, "to_string") else str(obj)
    block = f"\n== {title}\n{txt}"
    print(block); lines.append(block)

SUPPRESS_SMALL = False
import contextlib, io, html as _html
_show_en = show

def _fmt(v):
    # Display format of a table cell: thousands separator for counts, 4 significant digits for small values.
    if isinstance(v, (bool, np.bool_)) or v is None or (isinstance(v, float) and np.isnan(v)):
        return v
    if SUPPRESS_SMALL and isinstance(v, (int, np.integer)) and 0 < v < 10:
        return "<10"
    if isinstance(v, (int, np.integer)) or (isinstance(v, (float, np.floating)) and float(v).is_integer() and abs(v) >= 1000):
        return f"{int(v):,}"
    if isinstance(v, (float, np.floating)):
        return f"{v:.4g}" if abs(v) < 1 else f"{v:,.4f}".rstrip("0").rstrip(".")
    return v

def _render(title, obj):
    # In Jupyter: a heading and an HTML table with scroll; elsewhere: the text block.
    try:
        from IPython import get_ipython
        from IPython.display import HTML, display as _display
        if get_ipython() is None:              # plain Python (not a notebook)
            raise ImportError
    except ImportError:
        return print(lines_es[-1] if LANG == "es" else lines[-1])
    head = f"<p style='margin:14px 0 4px 0'><b>{_html.escape(str(title))}</b></p>"
    if isinstance(obj, (pd.Series, pd.DataFrame)):
        if isinstance(obj, pd.Series):          # rows are different quantities: each cell on its own
            df = obj.astype(object).map(_fmt).to_frame(name=translate("value", LANG))
        else:                                    # columns: the same number of decimals down each column
            df = obj.copy().astype(object)
            for k in range(df.shape[1]):
                col = obj.iloc[:, k]
                if pd.api.types.is_float_dtype(col) and not col.dropna().map(float.is_integer).all():
                    dec = max((len(f"{v:.4g}".split(".")[1]) if "." in f"{v:.4g}" and "e" not in f"{v:.4g}" else 0)
                              for v in col.dropna()) if col.notna().any() else 0
                    df.iloc[:, k] = col.map(lambda v, d=dec: "" if pd.isna(v) else f"{v:,.{d}f}")
                else:
                    df.iloc[:, k] = col.map(_fmt)
        table = df.to_html(na_rep="", border=0)
        _display(HTML(head + f"<div style='max-height:420px; overflow:auto'>{table}</div>"))
    else:
        _display(HTML(head + f"<pre style='margin:0'>{_html.escape(str(obj))}</pre>"))

def show(title, obj):
    # The English block is kept in `lines` (data/outputs_*.txt) and, in Spanish, the translated block in
    # `lines_es` (data/outputs_*_es.txt); on screen the table is shown as an HTML table.
    with contextlib.redirect_stdout(io.StringIO()):
        _show_en(title, obj)
        if LANG == "es":
            _show_en(translate(title, "es"), translate(obj, "es"))
            lines_es.append(lines.pop())
    _render(translate(title, LANG), translate(obj, LANG))
def yn(s):
    """Y -> 1, N -> 0, U or blank -> missing (decision M4-D3: no missingness indicators)."""
    s = s.astype("string")
    return pd.Series(np.where(s.eq("Y").fillna(False), 1.0, np.where(s.eq("N").fillna(False), 0.0, np.nan)),
                     index=s.index)

def coded(s, unknown):
    s = s.astype("Float64").astype(float)
    return s.where(~s.isin(unknown))

def dummies(X, s, levels):
    """One column per non-reference level; missing stays missing in every column."""
    for name, codes in levels:
        X[name] = np.where(s.isna(), np.nan, s.isin(codes).astype(float))

def make_predictors(d):
    """Predictors known at birth + 5 minutes (outcome card, sections 3 and 4; decisions M4-D2, M4-D3).
    Reference levels: White, high school or less, private insurance, no earlier living birth, first
    pregnancy, care in the first trimester, cephalic presentation, spontaneous vaginal delivery,
    hospital birth, physician attendant, singleton."""
    X = pd.DataFrame(index=d.index)
    # mother (marital status excluded: M4-D2)
    X["age"] = coded(d["MAGER"], [])
    dummies(X, coded(d["MRACEHISP"], [8]), [("race_black", [2]), ("race_hispanic", [7]), ("race_asian", [4]),
                                            ("race_other", [3, 5, 6])])
    dummies(X, coded(d["MEDUC"], [9]), [("educ_some_college", [4, 5]), ("educ_bachelor_plus", [6, 7, 8])])
    dummies(X, coded(d["MBSTATE_REC"], [3]), [("born_outside_us", [2])])
    dummies(X, coded(d["PAY_REC"], [9]), [("pay_medicaid", [1]), ("pay_self", [3]), ("pay_other", [4])])
    X["wic"] = yn(d["WIC"])
    # obstetric history
    dummies(X, coded(d["PRIORLIVE"], [99]), [("prior_living_1", [1]), ("prior_living_2plus", list(range(2, 99)))])
    X["prior_child_died"] = coded(d["PRIORDEAD"], [99]).ge(1).astype(float).where(coded(d["PRIORDEAD"], [99]).notna())
    X["prior_other_termination"] = coded(d["PRIORTERM"], [99]).ge(1).astype(float).where(coded(d["PRIORTERM"], [99]).notna())
    ilp = coded(d["ILP_R"], [999])
    ilp = ilp.where(~ilp.between(0, 3))           # 000-003: plural-delivery code, not an interval
    first = ilp == 888                            # first pregnancy: reference level, not unknown
    for name, lo, hi in [("interval_lt18", 4, 17), ("interval_18_59", 18, 59), ("interval_60plus", 60, 300)]:
        X[name] = np.where(ilp.isna(), np.nan, (~first & ilp.between(lo, hi)).astype(float))
    X["prior_cesarean"] = yn(d["RF_CESAR"])
    X["prior_preterm_birth"] = yn(d["RF_PPB"])
    # current pregnancy
    pc = coded(d["PRECARE"], [99])
    X["care_after_first_trimester"] = np.where(pc.isna(), np.nan, pc.between(4, 10).astype(float))
    X["no_prenatal_care"] = np.where(pc.isna(), np.nan, (pc == 0).astype(float))
    X["prenatal_visits"] = coded(d["PREVIS"], [99])
    bmi = d["BMI"].astype("Float64").astype(float)
    X["bmi"] = bmi.where(bmi.between(12, 80))
    X["weight_gain"] = coded(d["WTGAIN"], [99])
    X["smoked_during_pregnancy"] = yn(d["CIG_REC"])
    for name, col in [("pregestational_diabetes", "RF_PDIAB"), ("gestational_diabetes", "RF_GDIAB"),
                      ("chronic_hypertension", "RF_PHYPE"), ("gestational_hypertension", "RF_GHYPE"),
                      ("eclampsia", "RF_EHYPE"), ("infertility_treatment", "RF_INFT"),
                      ("gonorrhea", "IP_GON"), ("syphilis", "IP_SYPH"), ("chlamydia", "IP_CHLAM"),
                      ("hepatitis_b", "IP_HEPB"), ("hepatitis_c", "IP_HEPC")]:
        X[name] = yn(d[col])
    # labor and delivery
    for name, col in [("induction", "LD_INDL"), ("augmentation", "LD_AUGM"), ("antenatal_steroids", "LD_STER"),
                      ("antibiotics_in_labor", "LD_ANTB"), ("chorioamnionitis", "LD_CHOR"),
                      ("anesthesia", "LD_ANES")]:
        X[name] = yn(d[col])
    dummies(X, coded(d["ME_PRES"], [9]), [("presentation_breech", [2]), ("presentation_other", [3])])
    dummies(X, coded(d["ME_ROUT"], [9]), [("delivery_forceps", [2]), ("delivery_vacuum", [3]),
                                          ("delivery_cesarean", [4])])
    tr = d["ME_TRIAL"].astype("string")
    X["trial_of_labor"] = np.where(tr.eq("Y").fillna(False), 1.0,
                                   np.where(tr.isin(["N", "X"]).fillna(False), 0.0, np.nan))
    dummies(X, coded(d["BFACIL3"], [3, 9]), [("born_out_of_hospital", [2])])
    dummies(X, coded(d["ATTEND"], [9]), [("attendant_midwife", [3, 4]), ("attendant_other", [5])])
    # newborn
    dummies(X, coded(d["DPLURAL"], []), [("twin", [2]), ("triplet_or_more", [3, 4, 5])])
    X["male"] = (d["SEX"].astype("string") == "M").astype(float)
    X["gestational_age"] = coded(d["COMBGEST"], [99])
    X["birthweight"] = coded(d["BRTHWGT"], [9999])
    X["apgar5"] = coded(d["APGAR5"], [99])
    for name, col in [("anencephaly", "CA_ANEN"), ("spina_bifida", "CA_MNSB"), ("cyanotic_heart_disease", "CA_CCHD"),
                      ("diaphragmatic_hernia", "CA_CDH"), ("omphalocele", "CA_OMPH"), ("gastroschisis", "CA_GAST"),
                      ("limb_reduction", "CA_LIMB"), ("cleft_lip", "CA_CLEFT"), ("cleft_palate", "CA_CLPAL"),
                      ("hypospadias", "CA_HYPO")]:
        X[name] = yn(d[col])
    for name, col in [("down_syndrome", "CA_DOWN"), ("chromosomal_disorder", "CA_DISOR")]:
        s = d[col].astype("string")       # C confirmed, P pending, N no, U unknown
        X[name] = np.where(s.isin(["C", "P"]).fillna(False), 1.0, np.where(s.eq("N").fillna(False), 0.0, np.nan))
    return X.astype("float32")

REFERENCE = ["gestational_age", "birthweight"]       # reference model fixed in the outcome card
PREDICTORS = list(make_predictors(pd.read_parquet(OUT_DIR / "cohort_2023.parquet").head(5)).columns)
# The parameters of the contract table: block, NCHS source field and type (shown as a scrollable table)
SOURCE = {"age": "MAGER", "race_black": "MRACEHISP", "race_hispanic": "MRACEHISP", "race_asian": "MRACEHISP",
          "race_other": "MRACEHISP", "educ_some_college": "MEDUC", "educ_bachelor_plus": "MEDUC",
          "born_outside_us": "MBSTATE_REC", "pay_medicaid": "PAY_REC", "pay_self": "PAY_REC", "pay_other": "PAY_REC",
          "wic": "WIC", "prior_living_1": "PRIORLIVE", "prior_living_2plus": "PRIORLIVE", "prior_child_died": "PRIORDEAD",
          "prior_other_termination": "PRIORTERM", "interval_lt18": "ILP_R", "interval_18_59": "ILP_R",
          "interval_60plus": "ILP_R", "prior_cesarean": "RF_CESAR", "prior_preterm_birth": "RF_PPB",
          "care_after_first_trimester": "PRECARE", "no_prenatal_care": "PRECARE", "prenatal_visits": "PREVIS",
          "bmi": "BMI", "weight_gain": "WTGAIN", "smoked_during_pregnancy": "CIG_REC", "pregestational_diabetes": "RF_PDIAB",
          "gestational_diabetes": "RF_GDIAB", "chronic_hypertension": "RF_PHYPE", "gestational_hypertension": "RF_GHYPE",
          "eclampsia": "RF_EHYPE", "infertility_treatment": "RF_INFT", "gonorrhea": "IP_GON", "syphilis": "IP_SYPH",
          "chlamydia": "IP_CHLAM", "hepatitis_b": "IP_HEPB", "hepatitis_c": "IP_HEPC", "induction": "LD_INDL",
          "augmentation": "LD_AUGM", "antenatal_steroids": "LD_STER", "antibiotics_in_labor": "LD_ANTB",
          "chorioamnionitis": "LD_CHOR", "anesthesia": "LD_ANES", "presentation_breech": "ME_PRES",
          "presentation_other": "ME_PRES", "delivery_forceps": "ME_ROUT", "delivery_vacuum": "ME_ROUT",
          "delivery_cesarean": "ME_ROUT", "trial_of_labor": "ME_TRIAL", "born_out_of_hospital": "BFACIL3",
          "attendant_midwife": "ATTEND", "attendant_other": "ATTEND", "twin": "DPLURAL", "triplet_or_more": "DPLURAL",
          "male": "SEX", "gestational_age": "COMBGEST", "birthweight": "BRTHWGT", "apgar5": "APGAR5",
          "anencephaly": "CA_ANEN", "spina_bifida": "CA_MNSB", "cyanotic_heart_disease": "CA_CCHD",
          "diaphragmatic_hernia": "CA_CDH", "omphalocele": "CA_OMPH", "gastroschisis": "CA_GAST",
          "limb_reduction": "CA_LIMB", "cleft_lip": "CA_CLEFT", "cleft_palate": "CA_CLPAL", "hypospadias": "CA_HYPO",
          "down_syndrome": "CA_DOWN", "chromosomal_disorder": "CA_DISOR"}
BLOCK = {**{f: "mother" for f in ["MAGER", "MRACEHISP", "MEDUC", "MBSTATE_REC", "PAY_REC", "WIC"]},
         **{f: "obstetric history" for f in ["PRIORLIVE", "PRIORDEAD", "PRIORTERM", "ILP_R", "RF_CESAR", "RF_PPB"]},
         **{f: "current pregnancy" for f in ["PRECARE", "PREVIS", "BMI", "WTGAIN", "CIG_REC", "RF_PDIAB", "RF_GDIAB",
                                             "RF_PHYPE", "RF_GHYPE", "RF_EHYPE", "RF_INFT", "IP_GON", "IP_SYPH",
                                             "IP_CHLAM", "IP_HEPB", "IP_HEPC"]},
         **{f: "labor and delivery" for f in ["LD_INDL", "LD_AUGM", "LD_STER", "LD_ANTB", "LD_CHOR", "LD_ANES", "ME_PRES",
                                              "ME_ROUT", "ME_TRIAL", "BFACIL3", "ATTEND"]}}
CONTINUOUS = ["age", "prenatal_visits", "bmi", "weight_gain", "gestational_age", "birthweight", "apgar5"]
assert set(SOURCE) == set(PREDICTORS), "every parameter needs its source field"
predictor_table = pd.DataFrame({
    "block": [BLOCK.get(SOURCE[p], "newborn") for p in PREDICTORS],
    "NCHS field": [SOURCE[p] for p in PREDICTORS],
    "type": ["continuous" if p in CONTINUOUS else "binary (0/1)" for p in PREDICTORS],
    "reference model": ["yes" if p in REFERENCE else "" for p in PREDICTORS]},
    index=pd.Index(PREDICTORS, name="parameter"))
predictor_table.insert(0, "#", range(1, len(PREDICTORS) + 1))

def show_table(df, max_height=420):
    """Scrollable HTML table in Jupyter; plain text elsewhere."""
    try:
        from IPython import get_ipython
        from IPython.display import HTML, display as _display
        if get_ipython() is None:              # plain Python (not a notebook)
            raise ImportError
        _display(HTML(f'<div style="max-height:{max_height}px; overflow-y:auto">{df.to_html()}</div>'))
    except ImportError:
        print(df.to_string())

print(f"{len(PREDICTORS)} parameters from {predictor_table['NCHS field'].nunique()} NCHS fields "
      f"({len(CONTINUOUS)} continuous, {len(PREDICTORS) - len(CONTINUOUS)} binary)")
show_table(translate(predictor_table, LANG))
REBUILD = False   # True: rebuild the contract table even if its Parquet files exist (see the note above)

## E5 · Unit of analysis and overlap — one record per live birth

The unit is the live birth. The public-use files have no mother identifier, so the group is the birth itself and no group can fall on both sides of a split. Births of the same multiple delivery cannot be linked: a declared limitation.

| Result | What is done | In this model |
|---|---|---|
| One record per `unit_id` and `group_id = unit_id` in every cohort | On to S6 | Yes, in the 8 cohorts |
| Duplicated records | It stops: review the adapter (notebook 01) | — |

In [ ]:
# Stage S5 check: one record per live birth, and the group is the birth itself (no mother identifier in the
# public-use files), so no group can fall on both sides of a split.
u = []
for y in COHORTS:
    d = pd.read_parquet(OUT_DIR / f"cohort_{y}.parquet", columns=["unit_id", "group_id"])
    u.append({"cohort": y, "records": len(d), "unique unit_id": int(d["unit_id"].nunique()),
              "group_id = unit_id": bool((d["group_id"] == d["unit_id"]).all())})
unit = pd.DataFrame(u).set_index("cohort")
assert (unit["records"] == unit["unique unit_id"]).all() and unit["group_id = unit_id"].all()
display(translate(unit, LANG))

# Decision of stage S5
show("S5 decision: unit of analysis", pd.Series({
    "cohorts with one record per live birth": f"{int((unit['records'] == unit['unique unit_id']).sum())} of {len(unit)}",
    "group = the birth itself in every cohort": bool(unit["group_id = unit_id"].all()),
    "decision": "continue to S6 (no group can fall on both sides of a split)"}))

## E6 · Prediction point and information window — 5 minutes after birth

Time zero: birth. Prediction point: 5 minutes later, when the 5-minute Apgar exists. Only what is known by then enters; what happens later or describes the newborn's course (intensive care, ventilation, 10-minute Apgar, transfer, breastfeeding) and the mother's delivery morbidity are excluded.

| Result | What is done | In this model |
|---|---|---|
| No field after the prediction point is read | On to S7 | None of the 15 later fields |
| Any is read | It stops: remove it from the list before building the table | — |

In [ ]:
# Stage S6 check: prediction point = birth + 5 minutes. The NCHS fields read for the predictors (FIELDS) must
# not include anything recorded after that point or describing the newborn's course.
FIELDS = ["MAGER", "MRACEHISP", "MEDUC", "MBSTATE_REC", "PAY_REC", "WIC", "PRIORLIVE", "PRIORDEAD", "PRIORTERM",
          "ILP_R", "RF_CESAR", "RF_PPB", "PRECARE", "PREVIS", "BMI", "WTGAIN", "CIG_REC", "RF_PDIAB", "RF_GDIAB",
          "RF_PHYPE", "RF_GHYPE", "RF_EHYPE", "RF_INFT", "IP_GON", "IP_SYPH", "IP_CHLAM", "IP_HEPB", "IP_HEPC",
          "LD_INDL", "LD_AUGM", "LD_STER", "LD_ANTB", "LD_CHOR", "LD_ANES", "ME_PRES", "ME_ROUT", "ME_TRIAL",
          "BFACIL3", "ATTEND", "DPLURAL", "SEX", "COMBGEST", "BRTHWGT", "APGAR5", "CA_ANEN", "CA_MNSB", "CA_CCHD",
          "CA_CDH", "CA_OMPH", "CA_GAST", "CA_LIMB", "CA_CLEFT", "CA_CLPAL", "CA_HYPO", "CA_DOWN", "CA_DISOR"]
AFTER_PREDICTION_POINT = {"AB_AVEN1": "assisted ventilation", "AB_AVEN6": "ventilation > 6 hours", "AB_NICU": "NICU admission",
                          "AB_SURF": "surfactant", "AB_ANTI": "antibiotics for the newborn", "AB_SEIZ": "seizures",
                          "APGAR10": "10-minute Apgar", "ITRAN": "infant transferred", "ILIVE": "alive at report",
                          "BFED": "breastfed at discharge", "MM_MTR": "maternal transfusion", "MM_PLAC": "perineal laceration",
                          "MM_RUPT": "ruptured uterus", "MM_UHYST": "hysterectomy", "MM_AICU": "maternal ICU admission"}
window = pd.DataFrame({"field": list(AFTER_PREDICTION_POINT), "content": list(AFTER_PREDICTION_POINT.values()),
                       "read for the predictors": [f in FIELDS for f in AFTER_PREDICTION_POINT]}).set_index("field")
assert not window["read for the predictors"].any(), "a field recorded after the prediction point is read"
print(f"{len(FIELDS)} NCHS fields read for the predictors; none of the {len(window)} fields recorded after birth + 5 minutes")
display(translate(window, LANG))

# Decision of stage S6
show("S6 decision: information window", pd.Series({
    "NCHS fields read for the predictors": len(FIELDS),
    "fields recorded after birth + 5 minutes that are read": int(window["read for the predictors"].sum()),
    "decision": "continue to S7 (only information known at birth + 5 minutes)"}))

## E7 · Predictors — contract table and coverage of the 71 parameters

Mother, obstetric history, current pregnancy, labor and delivery, and newborn (5-minute Apgar included). NCHS imputation flags are kept for quality control only.

| Result | What is done | In this model |
|---|---|---|
| Missing ≤ 3% | Enter with median imputation (M4-D3) | 68 of 71 (the 3 interpregnancy-interval parameters have 12.7%) |
| Missing > 10% in a predictor | Kept only if decided in the audit; complete records go as a sensitivity analysis | Interpregnancy interval 12.7% (decided in S4) |

In [ ]:
# Contract table, one file per cohort (not versioned: derived from NCHS data)
CONTRACT_DIR.mkdir(parents=True, exist_ok=True)
for y in COHORTS:
    final = CONTRACT_DIR / f"cohort_{y}.parquet"
    if final.exists() and not REBUILD:
        continue
    d = pd.read_parquet(OUT_DIR / f"cohort_{y}.parquet",
                        columns=["unit_id", "group_id", "RESTATUS", "COMBGST_IMP", "BWTIMP", "label_main",
                                 "label_early_neonatal", "d_age_days"] + [f for f in FIELDS if f != "COMBGEST"] + ["COMBGEST"])
    d = d[(d["RESTATUS"] < 4) & d["COMBGEST"].between(22, 47)].reset_index(drop=True)
    X = make_predictors(d)
    c = pd.DataFrame({"unit_id": d["unit_id"], "group_id": d["group_id"], "period": np.int16(y)})
    c["label_main"] = d["label_main"].to_numpy(bool)
    died_day0_1 = (c["label_main"] & d["d_age_days"].le(1).fillna(False)).to_numpy(bool)
    c["at_risk_48h"] = ~died_day0_1                                     # sensitivity (a), M4-D1
    c["label_day2_27"] = c["label_main"] & c["at_risk_48h"]
    c["label_early_neonatal"] = d["label_early_neonatal"].to_numpy(bool)  # sensitivity (b)
    c["ge28"] = (d["COMBGEST"] >= 28).to_numpy(bool)                     # sensitivity (c)
    c["ga_imputed"] = (d["COMBGST_IMP"].astype("string") == "1").fillna(False).to_numpy(bool)   # data quality only
    c["bw_imputed"] = (d["BWTIMP"].astype("string") == "1").fillna(False).to_numpy(bool)        # data quality only
    c = pd.concat([c, X], axis=1)
    c["n_unknown"] = X.isna().sum(axis=1).astype("int8")      # leakage check only, never a predictor
    tmp = final.with_suffix(".parquet.tmp")
    c.to_parquet(tmp, index=False); tmp.replace(final)
    print(f"cohort {y}: {len(c)} births written")
    del d, X, c

rows, n_total = {}, 0
for y in COHORTS:
    X = pd.read_parquet(CONTRACT_DIR / f"cohort_{y}.parquet", columns=PREDICTORS)
    n_total += len(X)
    for p in PREDICTORS:
        r = rows.setdefault(p, [0, 0.0, 0])
        r[0] += int(X[p].isna().sum()); r[1] += float(X[p].sum()); r[2] += int(X[p].notna().sum())
    del X
cov = pd.DataFrame({"pct_missing": {p: round(100 * rows[p][0] / n_total, 2) for p in PREDICTORS},
                    "mean_or_pct_yes": {p: round(rows[p][1] / rows[p][2] * (1 if p in
                        ["age", "prenatal_visits", "bmi", "weight_gain", "gestational_age", "birthweight", "apgar5"] else 100), 2)
                        for p in PREDICTORS}})
show(f"Q-P1 Contract table: {n_total} births, {len(PREDICTORS)} parameters; % missing and mean (continuous) or % yes, whole population", cov)

# Decision of stage S7: which parameters enter and how their missing values are handled
pd.set_option("display.max_colwidth", None)
low = cov[cov["pct_missing"] <= 3]
high = cov[cov["pct_missing"] > 3]
show("S7 decision: missing values of the parameters", pd.DataFrame({
    "parameters": [len(low), len(high)],
    "highest % missing": [float(low["pct_missing"].max()), float(high["pct_missing"].max()) if len(high) else None],
    "which": [f"all others", ", ".join(high.index) or "none"],
    "decision": ["enter with median imputation, no missingness indicators (M4-D3)",
                 "enter: interpregnancy interval kept in the audit (S4, M4-D2 table); complete records as a sensitivity analysis"]},
    index=["missing <= 3%", "missing > 3%"]))

## P2 · Leakage gate — nothing after the prediction point and signal of the unknowns

Design checks and AUROC of the number of unknown fields in the development cohorts.

| Result | What is done | In this model |
|---|---|---|
| All checks = True | G2 passes: on to G3 | Passes |
| Any = False | It stops: remove the leaking predictor and rebuild the table | — |
| AUROC of the unknowns clearly > 0.50 | M4-D3 and the complete-record sensitivity are kept; the native-XGBoost diagnostic is checked in notebook 04 | 0.601 (0.571 at ≥ 28 weeks) |

In [ ]:
# Gate G2 (leakage): design checks
EXCLUDED = ["label_main", "label_day2_27", "label_early_neonatal", "at_risk_48h", "ge28", "n_unknown",
            "ga_imputed", "bw_imputed", "marital_status",
            # recorded after the prediction point or reflecting the newborn's course (outcome card, section 3)
            "assisted_ventilation", "ventilation_6h", "nicu_admission", "surfactant", "newborn_antibiotics",
            "seizures", "apgar10", "transferred", "alive_at_report", "breastfed", "maternal_transfusion",
            "perineal_laceration", "uterine_rupture", "hysterectomy", "maternal_icu"]
checks = {
    "no outcome, landmark, death-certificate or post-prediction field among the predictors": not set(PREDICTORS) & set(EXCLUDED),
    "no missingness or imputation indicator among the predictors (M4-D3)":
        not any(k in p for p in PREDICTORS for k in ("missing", "unknown", "imputed")),
    "marital status excluded (M4-D2)": not any("marital" in p or "dmar" in p.lower() for p in PREDICTORS),
    "unit of analysis is the live birth and the group is the unit itself (no mother identifier)": True,
}
show("Gate G2: design checks", pd.Series(checks))
assert all(checks.values())

# G2 leakage check: AUROC of the number of unknown predictor fields alone (development cohorts)
def auroc_from_counts(neg, pos):
    """Exact AUROC (ties count one half) of an ordinal score from counts per score value."""
    neg, pos = np.asarray(neg, float), np.asarray(pos, float)
    below = np.concatenate([[0], np.cumsum(neg)[:-1]])
    return float(((below + neg / 2) * pos).sum() / (neg.sum() * pos.sum()))

res = {}
for name, cond in [("population", None), ("ge28_weeks", "ge28")]:
    tab = None
    for y in DEV:
        c = pd.read_parquet(CONTRACT_DIR / f"cohort_{y}.parquet", columns=["n_unknown", "label_main", "ge28"])
        if cond:
            c = c[c[cond]]
        t = pd.crosstab(c["n_unknown"], c["label_main"])
        tab = t if tab is None else tab.add(t, fill_value=0)
    tab = tab.reindex(columns=[False, True], fill_value=0).astype(np.int64)
    res[name] = round(auroc_from_counts(tab[False], tab[True]), 3)
    if name == "population":
        show("G2 leakage check: births by number of unknown predictor fields (development cohorts)",
             pd.DataFrame({"births": tab.sum(axis=1), "pct": (100 * tab.sum(axis=1) / tab.values.sum()).round(2)}))
show("G2 leakage check: AUROC of the number of unknown fields alone (development cohorts)", pd.Series(res))

# Decision of gate G2
above = res["population"] > 0.55            # reading criterion, as in notebook 02
show("G2 decision", pd.Series({
    "design checks passed": f"{sum(checks.values())} of {len(checks)}",
    "AUROC of the number of unknowns (population / 28 weeks or more)": f"{res['population']:.3f} / {res['ge28_weeks']:.3f}",
    "clearly above 0.50 (AUROC > 0.55)": bool(above),
    "decision": ("G2 passes; M4-D3 and the complete-record sensitivity analysis are kept; the native-XGBoost "
                 "diagnostic is checked in notebook 04") if above else "G2 passes; no special handling of unknowns"}))

## P3 · Sample-size gate — Riley et al. criteria in two steps

Planning with an assumed Cox-Snell R² of 15% of its maximum and reassessment with the observed out-of-fold R² of a logistic regression; for the full and the reference model.

| Result | What is done | In this model |
|---|---|---|
| Births and events available ≥ required in both steps | G3 passes: preliminary run (notebook 04) | Passes: 22.6 million and 62,713 events vs 37,427 and 104 required |
| Not enough | Feasibility and R1 to the next rung; if the model already meets G4, one sample-size reduction | — |
| Events per parameter ≥ 20 | Primary learner = the best of a preliminary run | 883 per parameter |

In [ ]:
# Gate G3 (sample size), two steps: planning with an assumed Cox-Snell R2 of 15% of its
# maximum, then reassessment with the observed out-of-fold R2 of a cross-validated logistic regression.
# Development cohorts only. All events and a random fraction of non-events, weighted, so that the
# log-likelihood estimates that of the full data. Parameters = one-hot columns, without the intercept.
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

FRACTION = 0.03
SETS = {"main model (all predictors)": PREDICTORS, "reference model (gestational age + birthweight)": REFERENCE}
rng = np.random.default_rng(SEED)
parts, n_dev, e_dev = [], 0, 0
for y in DEV:
    c = pd.read_parquet(CONTRACT_DIR / f"cohort_{y}.parquet", columns=PREDICTORS + ["label_main"])
    n_dev += len(c); e_dev += int(c["label_main"].sum())
    keep = c["label_main"].to_numpy() | (rng.random(len(c)) < FRACTION)
    parts.append(c[keep]); del c
s = pd.concat(parts, ignore_index=True); del parts
y_s = s.pop("label_main").to_numpy(bool)
w_s = np.where(y_s, 1.0, 1.0 / FRACTION)
phi = e_dev / n_dev
L0n = phi * np.log(phi) + (1 - phi) * np.log(1 - phi)
R2max = 1 - np.exp(2 * L0n)

def riley(P, r2, n_avail, shrink=0.9):
    n1 = P / ((shrink - 1) * np.log(1 - r2 / shrink))
    s2 = r2 / (r2 + 0.05 * R2max)
    n2 = P / ((s2 - 1) * np.log(1 - r2 / s2))
    n3 = (1.96 / 0.05) ** 2 * phi * (1 - phi)
    req = int(np.ceil(max(n1, n2, n3)))
    smax = max(shrink, s2)
    return {"n required (shrinkage)": int(np.ceil(n1)), "n required (optimism)": int(np.ceil(n2)),
            "n required (overall risk)": int(np.ceil(n3)), "n required": req,
            "events required": int(np.ceil(req * phi)),
            "parameters supported": int(np.floor(n_avail * (smax - 1) * np.log(1 - r2 / smax))),
            "gate G3": "PASSED" if n_avail >= req and e_dev >= req * phi else "NOT PASSED"}

out = {}
for name, cols in SETS.items():
    P = len(cols)
    plan = riley(P, 0.15 * R2max, n_dev)
    oof = np.zeros(len(s))
    for tr, te in StratifiedKFold(5, shuffle=True, random_state=SEED).split(s, y_s):
        m = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(C=1e6, max_iter=2000))
        m.fit(s.iloc[tr][cols], y_s[tr], logisticregression__sample_weight=w_s[tr])
        oof[te] = m.predict_proba(s.iloc[te][cols])[:, 1]
    p = np.clip(oof, 1e-12, 1 - 1e-12)
    n = float(w_s.sum())
    L1 = float(np.sum(w_s * (y_s * np.log(p) + (~y_s) * np.log(1 - p))))
    R2 = 1 - np.exp(2 * (n * L0n - L1) / n)
    obs = riley(P, R2, n_dev)
    out[name] = {"candidate parameters": P, "events per parameter": round(e_dev / P, 1),
                 "assumed R2 (15% of max)": round(0.15 * R2max, 5),
                 **{f"planning: {k}": v for k, v in plan.items()},
                 "observed Cox-Snell R2 (cross-validated)": round(R2, 5), "% of maximum": round(100 * R2 / R2max, 1),
                 **{f"reassessment: {k}": v for k, v in obs.items()}}
show("Gate G3: data (development cohorts 2016-2021)",
     pd.Series({"development births": n_dev, "development events": e_dev, "prevalence": round(phi, 5),
                "Cox-Snell R2 max": round(R2max, 5), "sample used (all events + 3% of non-events)": len(s)}, dtype=object))
show("Gate G3: sample size (Riley et al.), planning and reassessment", pd.DataFrame(out))
show("Primary learner rule (events per parameter >= 20: best base learner of a preliminary run)",
     "events per parameter = %.1f" % (e_dev / len(PREDICTORS)))

# Decision of gate G3
g3d = pd.DataFrame(out)
passed = all(g3d.loc[k].eq("PASSED").all() for k in ["planning: gate G3", "reassessment: gate G3"])
epp = e_dev / len(PREDICTORS)
show("G3 decision", pd.Series({
    "gate G3, planning and reassessment (main and reference)": "PASSED" if passed else "NOT PASSED",
    "events available / required (main model, reassessment)": f"{e_dev:,} / {g3d.iloc[:, 0]['reassessment: events required']:,}",
    "events per parameter": round(epp, 1),
    "decision": ("G3 passes: preliminary run of the base learners (notebook 04); with 20 or more events per "
                 "parameter the primary learner is the best one of that run") if passed and epp >= 20 else
                ("G3 passes: preliminary run (notebook 04)" if passed else "G3 not passed: feasibility and R1 to the next rung")}))

write_outputs(OUT_FILE, "Predictor table and gates G2 and G3 of the neonatal model (aggregate output only)")
print("written:", OUT_FILE)

## Notebook summary

Result of gates G2 and G3.

In [ ]:
# Summary of notebook 03
g3 = pd.DataFrame(out)
summary = pd.Series({
    "births in the contract table": n_total, "parameters": len(PREDICTORS),
    "largest % missing (predictor)": f"{cov['pct_missing'].max():.2f} ({cov['pct_missing'].idxmax()})",
    "gate G2, design checks": "passes" if all(checks.values()) else "DOES NOT PASS",
    "AUROC of the number of unknowns (development)": res["population"],
    "development births / events": f"{n_dev:,} / {e_dev:,}",
    "events per parameter": round(e_dev / len(PREDICTORS), 1),
    "gate G3, planning (main / reference)": f"{g3.loc['planning: gate G3'].iloc[0]} / {g3.loc['planning: gate G3'].iloc[1]}",
    "gate G3, reassessment (main / reference)": f"{g3.loc['reassessment: gate G3'].iloc[0]} / {g3.loc['reassessment: gate G3'].iloc[1]}",
    "observed Cox-Snell R2 (main / reference)": f"{g3.loc['observed Cox-Snell R2 (cross-validated)'].iloc[0]} / {g3.loc['observed Cox-Snell R2 (cross-validated)'].iloc[1]}"},
    name="value")
translate(summary.to_frame(), LANG)